In [366]:
from numba import njit
import numpy as np

In [367]:
# 1. Обычная функция:
def f(a):
    return a + 1

print("type(f):", type(f))
print("f.__code__:", f.__code__)
print("f.__code__.co_varnames:", f.__code__.co_varnames)

type(f): <class 'function'>
f.__code__: <code object f at 0x115e3c210, file "/var/folders/bm/j41t02r53h3b5lv3gfmlt_j40000gn/T/ipykernel_65933/2840393523.py", line 2>
f.__code__.co_varnames: ('a',)


In [368]:
# 2. Навешиваем декоратор
jf = njit(f)

print("type(jf):", type(jf))
print("jf.py_func:", jf.py_func)
print("jf.signatures:", jf.signatures)
print("jf.overloads:", jf.overloads)

type(jf): <class 'numba.core.registry.CPUDispatcher'>
jf.py_func: <function f at 0x1156ac930>
jf.signatures: []
jf.overloads: OrderedDict()


In [ ]:
# 3. Первый вызов - компиляция
arg_value = 1

print("result:", jf(arg_value))
print("jf.signatures:", jf.signatures)
print("jf.overloads keys:", list(jf.overloads.keys()))

result: 10.0
jf.signatures: [(float64,)]
jf.overloads keys: [(float64,)]


In [370]:
# 4. Достаём объект результата компиляции
sig = jf.signatures[0]
cres = jf.overloads[sig]

print("type(cres):", type(cres))
print("cres.signature:", cres.signature)
print("cres.entry_point:", cres.entry_point)
print("cres.library:", cres.library)

type(cres): <class 'numba.core.compiler.CompileResult'>
cres.signature: (float64,) -> float64
cres.entry_point: <built-in method f of _dynfunc._Closure object at 0x115d0ece0>
cres.library: <Library 'f' at 0x115638cb0>


In [371]:
# 5. LLVM IR и ассемблер — это и есть низкоуровневый результат компиляции
print("\n6) LLVM IR, наша функция f:")
llvm = jf.inspect_llvm(sig)
llvm_str = "\n".join(str(llvm).splitlines()[11:17])
print("...\n" + llvm_str + "\n...")

print("\n7) Assembly, наша функция f:")
asm = jf.inspect_asm(sig)
asm_str = "\n".join(str(asm).splitlines()[4:9])
print("...\n" + asm_str + "\n...")

print("\n8) Второй вызов — уже без компиляции, используется готовая версия:")
print("result:", jf(arg_value))



6) LLVM IR, наша функция f:
...
define noundef i32 @_ZN8__main__1fB3v66B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEd(ptr noalias nocapture writeonly initializes((0, 8)) %retptr, ptr noalias nocapture readnone %excinfo, double %arg.a) local_unnamed_addr #0 {
B0:
  %.6 = fadd double %arg.a, 1.000000e+00
  store double %.6, ptr %retptr, align 8
  ret i32 0
}
...

7) Assembly, наша функция f:
...
__ZN8__main__1fB3v66B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEd:
	fmov	d1, #1.00000000
	fadd	d0, d0, d1
	str	d0, [x0]
	mov	w0, #0
...

8) Второй вызов — уже без компиляции, используется готовая версия:
result: 10.0
